# 🫁 PulmoScan AI — Chest CT Lung-Cancer Classification

**Goal:** train the most accurate model we can on the 4-class Chest-CT dataset
and export a checkpoint the FastAPI service can serve directly.

**Classes:** `adenocarcinoma` · `large.cell.carcinoma` · `normal` · `squamous.cell.carcinoma`

**Recipe:**
1. EDA — class balance, image sizes, color modes, sample grids
2. Data prep — canonical class mapping, augmentation, class weights
3. Transfer learning — pretrained CNN backbone + fresh head
4. **Two-phase training** — (1) train head with frozen backbone, (2) fine-tune end to end
5. Honest evaluation on the held-out **test** split — confusion matrix, per-class metrics
6. Export a self-describing checkpoint (`model.pt`) for serving

This notebook reuses the project package (`pulmoscan.models`, `pulmoscan.utils.data`)
so what we explore here is exactly what `dvc repro` / `python main.py` runs.

In [ ]:
import os
import sys

# Run from the project root so `Data/` and the `pulmoscan` package resolve.
if os.path.basename(os.getcwd()) == "research":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import warnings
warnings.filterwarnings("ignore")

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader

from pulmoscan.models import build_model, get_eval_transforms, get_train_transforms
from pulmoscan.utils.common import get_device, set_seed
from pulmoscan.utils.data import (
    build_datasets,
    build_test_dataset,
    canonical_class,
    compute_class_weights,
)

sns.set_theme(style="whitegrid")
DATA_ROOT = "Data"
IMAGE_SIZE = 224
SEED = 42
set_seed(SEED)
device = get_device()
print("Device:", device)

## 1. Exploratory Data Analysis

In [ ]:
# Count images per split and canonical class.
rows = []
for split in ["train", "valid", "test"]:
    split_dir = Path(DATA_ROOT) / split
    for class_dir in sorted(split_dir.iterdir()):
        if not class_dir.is_dir():
            continue
        n = sum(1 for p in class_dir.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"})
        rows.append({"split": split, "class": canonical_class(class_dir.name), "count": n})

df = pd.DataFrame(rows)
pivot = df.pivot_table(index="class", columns="split", values="count", aggfunc="sum", fill_value=0)
pivot = pivot[["train", "valid", "test"]]
pivot["total"] = pivot.sum(axis=1)
print("Total images:", pivot["total"].sum())
pivot

In [ ]:
# Class distribution per split.
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
pivot[["train", "valid", "test"]].plot(kind="bar", ax=ax[0])
ax[0].set_title("Image count per class and split")
ax[0].set_ylabel("images")
ax[0].tick_params(axis="x", rotation=20)

pivot["total"].plot(kind="pie", ax=ax[1], autopct="%1.1f%%", ylabel="")
ax[1].set_title("Overall class balance")
plt.tight_layout()
plt.show()

In [ ]:
# Image dimensions and color modes across the whole dataset.
widths, heights, modes = [], [], Counter()
for p in Path(DATA_ROOT).rglob("*"):
    if p.suffix.lower() not in {".png", ".jpg", ".jpeg"}:
        continue
    with Image.open(p) as im:
        widths.append(im.size[0]); heights.append(im.size[1]); modes[im.mode] += 1

print("Color modes:", dict(modes), "(all converted to RGB at load time)")
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].scatter(widths, heights, s=8, alpha=0.3)
ax[0].axvline(IMAGE_SIZE, color="r", ls="--"); ax[0].axhline(IMAGE_SIZE, color="r", ls="--")
ax[0].set(title="Image dimensions (red = 224 target)", xlabel="width", ylabel="height")
ax[1].hist(widths, bins=40, alpha=0.6, label="width")
ax[1].hist(heights, bins=40, alpha=0.6, label="height")
ax[1].legend(); ax[1].set(title="Dimension distribution", xlabel="pixels")
plt.tight_layout(); plt.show()

In [ ]:
# Sample grid: 4 images per class from the train split.
classes = sorted({canonical_class(d.name) for d in (Path(DATA_ROOT) / "train").iterdir() if d.is_dir()})
fig, axes = plt.subplots(len(classes), 4, figsize=(12, 3 * len(classes)))
for r, cls in enumerate(classes):
    folder = next(d for d in (Path(DATA_ROOT) / "train").iterdir() if canonical_class(d.name) == cls)
    imgs = [p for p in folder.iterdir() if p.suffix.lower() in {".png", ".jpg", ".jpeg"}][:4]
    for c, img_path in enumerate(imgs):
        axes[r, c].imshow(Image.open(img_path).convert("RGB"))
        axes[r, c].axis("off")
    axes[r, 0].set_ylabel(cls, rotation=0, ha="right", va="center", fontsize=11)
fig.suptitle("Sample CT scans per class", y=1.0)
plt.tight_layout(); plt.show()

### EDA takeaways
- **Imbalanced**: large-cell carcinoma is the minority class → use class weights.
- **Sizes vary widely** → resize to 224×224.
- **Mixed color modes** (RGBA/RGB/grayscale) → the loader converts everything to RGB.
- Pre-split into train/valid/test → train on train+valid, report on **test**.

## 2. Data preparation

In [ ]:
train_ds, val_ds, class_names, train_targets = build_datasets(
    data_root=DATA_ROOT, image_size=IMAGE_SIZE, augmentation=True, val_split=0.2, seed=SEED,
)
test_ds = build_test_dataset(DATA_ROOT, IMAGE_SIZE)
NUM_CLASSES = len(class_names)
print("Classes:", class_names)
print(f"train={len(train_ds)}  val={len(val_ds)}  test={len(test_ds)}")

BATCH_SIZE = 32
pin = device.type == "cuda"
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=pin)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=pin)

class_weights = compute_class_weights(train_targets, NUM_CLASSES).to(device)
print("Class weights:", {c: round(w, 3) for c, w in zip(class_names, class_weights.cpu().tolist())})

## 3. Model — transfer learning

In [ ]:
# Try resnet50 by default. Swap BACKBONE to compare convnext_tiny / efficientnet_v2_s.
BACKBONE = "resnet50"
model = build_model(BACKBONE, num_classes=NUM_CLASSES, pretrained=True, freeze_backbone=True).to(device)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"{BACKBONE}: {trainable:,} trainable / {total:,} total params (phase 1 = head only)")

## 4. Two-phase training

In [ ]:
LABEL_SMOOTHING = 0.1
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=LABEL_SMOOTHING)
history = {"phase": [], "epoch": [], "train_loss": [], "val_loss": [], "val_acc": []}
best_acc, best_state = 0.0, None


def set_backbone_frozen(model, frozen: bool):
    for p in model.parameters():
        p.requires_grad = not frozen
    head = getattr(model, "fc", None) or getattr(model, "classifier", None)
    for p in head.parameters():
        p.requires_grad = True


def run_phase(model, name, epochs, lr):
    global best_acc, best_state
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(epochs, 1))
    for epoch in range(1, epochs + 1):
        model.train()
        tl = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = criterion(model(x), y)
            loss.backward(); opt.step()
            tl += loss.item() * x.size(0)
        tl /= len(train_loader.dataset)

        model.eval(); vl, correct = 0.0, 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                vl += criterion(out, y).item() * x.size(0)
                correct += (out.argmax(1) == y).sum().item()
        vl /= len(val_loader.dataset); va = correct / len(val_loader.dataset)
        sched.step()
        history["phase"].append(name); history["epoch"].append(epoch)
        history["train_loss"].append(tl); history["val_loss"].append(vl); history["val_acc"].append(va)
        flag = ""
        if va >= best_acc:
            best_acc = va; best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            flag = "  <- best"
        print(f"[{name}] {epoch:2d}/{epochs} | train {tl:.3f} | val {vl:.3f} | acc {va:.3f}{flag}")


print("=== Phase 1: head only ===")
set_backbone_frozen(model, True)
run_phase(model, "head", epochs=15, lr=1e-3)

print("\n=== Phase 2: fine-tune end to end ===")
set_backbone_frozen(model, False)
run_phase(model, "finetune", epochs=10, lr=5e-5)

print(f"\nBest validation accuracy: {best_acc:.4f}")

In [ ]:
# Training curves.
h = pd.DataFrame(history)
h["step"] = range(1, len(h) + 1)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(h["step"], h["train_loss"], label="train")
ax[0].plot(h["step"], h["val_loss"], label="val")
ax[0].set(title="Loss", xlabel="epoch (cumulative)"); ax[0].legend()
ax[1].plot(h["step"], h["val_acc"], color="green")
ax[1].set(title="Validation accuracy", xlabel="epoch (cumulative)")
phase2_start = (h["phase"] == "finetune").idxmax() + 1
for a in ax:
    a.axvline(phase2_start, color="gray", ls="--", alpha=0.7)
plt.tight_layout(); plt.show()

## 5. Evaluation on the held-out test set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

model.load_state_dict(best_state)
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for x, y in test_loader:
        out = model(x.to(device))
        y_pred.extend(out.argmax(1).cpu().tolist())
        y_true.extend(y.tolist())

acc = np.mean(np.array(y_true) == np.array(y_pred))
print(f"TEST accuracy: {acc:.4f}\n")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(f"Confusion matrix (test acc {acc:.3f})")
plt.tight_layout(); plt.show()

## 6. Export the serving checkpoint

In [ ]:
# Self-describing checkpoint — identical format to the training pipeline,
# loadable by app/services/inference.py with no external metadata.
Path("artifacts/training").mkdir(parents=True, exist_ok=True)
ckpt_path = "artifacts/training/model.pt"
torch.save(
    {
        "state_dict": best_state,
        "backbone": BACKBONE,
        "num_classes": NUM_CLASSES,
        "class_names": class_names,
        "image_size": IMAGE_SIZE,
    },
    ckpt_path,
)
with open("scores.json", "w") as f:
    json.dump({"test_accuracy": float(acc), "best_val_accuracy": float(best_acc)}, f, indent=2)
print("Saved", ckpt_path)
print("Serve it:  MODEL_PATH=artifacts/training/model.pt uvicorn app.main:app")

## 7. Conclusions & how to push accuracy further

- The two-phase recipe (frozen head → full fine-tune) is the reliable baseline.
- To squeeze out more accuracy, try in order:
  1. **Backbone swap** — set `BACKBONE = "convnext_tiny"` or `"efficientnet_v2_s"` and re-run.
  2. **Longer fine-tuning** with a smaller LR and early stopping.
  3. **Stronger augmentation** (RandAugment, MixUp) — medical CT benefits from mild geometric aug.
  4. **Test-time augmentation** (average predictions over flips).
- The winning settings map 1:1 onto `params.yaml`, so `dvc repro` reproduces this run headlessly.